# Asteroid Dataset: Feature Dictionary & Preprocessing Strategy

##1 Dataset Overview
This dataset contains orbital and physical characteristics of asteroids obtained from NASA/JPL databases.
The primary objective is to predict **PHA (Potentially Hazardous Asteroid)**.

> **Target Variable:** `pha` — Binary classification (0 = Safe, 1 = Hazardous)

A PHA is typically defined based on its **Minimum Orbit Intersection Distance (MOID)** with Earth and its **Absolute Magnitude (H)** (which is a proxy for size).

---

## 2 Feature Descriptions (What They Mean)

### Orbital Parameters (Highly Predictive)
These physically describe the asteroid’s orbit around the sun:
* **`a`** : Semi-major axis (average distance from the Sun)
* **`e`** : Orbital eccentricity (how elliptical the orbit is)
* **`i`** : Inclination (tilt of the orbit relative to the ecliptic plane)
* **`q`** : Perihelion distance (closest approach to the Sun)
* **`ad`** : Aphelion distance (farthest distance from the Sun)
* **`om`** : Longitude of the ascending node
* **`w`** : Argument of perihelion
* **`ma`** : Mean anomaly
* **`n`** : Mean motion
* **`tp`** : Time of perihelion passage
* **`per` / `per_y`** : Orbital period in days / years
* **`moid` / `moid_ld`** : Minimum Orbit Intersection Distance with Earth (in AU / Lunar Distances) **Critical feature**
* **`rms`** : Orbit solution uncertainty

###  Physical Characteristics
* **`H`** : Absolute magnitude (brightness, used to estimate size)  **Critical feature**

### Measurement Uncertainties
* **`sigma_*` columns** (e.g., `sigma_e`, `sigma_a`): Represent the margin of error/uncertainty for the orbital measurements. Highly uncertain orbits can sometimes correlate with hazard potential.

---



In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
from google.colab import drive
drive.mount('/content/drive')

file_path = '/content/drive/My Drive/Colab Notebooks/asteroid_dataset.csv'
df = pd.read_csv(file_path)

Mounted at /content/drive


/tmp/ipykernel_1227/2203525960.py:5: DtypeWarning: Columns (3,4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


In [5]:
df.columns

Index(['id', 'spkid', 'full_name', 'pdes', 'name', 'prefix', 'neo', 'pha', 'H',
       'diameter', 'albedo', 'diameter_sigma', 'orbit_id', 'epoch',
       'epoch_mjd', 'epoch_cal', 'equinox', 'e', 'a', 'q', 'i', 'om', 'w',
       'ma', 'ad', 'n', 'tp', 'tp_cal', 'per', 'per_y', 'moid', 'moid_ld',
       'sigma_e', 'sigma_a', 'sigma_q', 'sigma_i', 'sigma_om', 'sigma_w',
       'sigma_ma', 'sigma_ad', 'sigma_n', 'sigma_tp', 'sigma_per', 'class',
       'rms'],
      dtype='object')

In [6]:
names = df['name']     # save separately
df = df.drop(columns=['name'])

## Dropped Columns & Justification (Why We Removed Them)

To prepare the data for a Neural Network, several columns were removed to prevent data leakage, overfitting, and noise.

### 1. Identifiers & Text (Zero Predictive Power)
**Dropped:** `id`, `spkid`, `full_name`, `pdes`, `name`, `prefix`, `orbit_id`
* **Why:** Neural networks look for mathematical patterns, not names. Including high-cardinality IDs forces the model to memorize data (overfitting) rather than generalize. *(Note: `name` is kept in a separate variable just for final result mapping, but excluded from training).*

### 2. Data Leakage (Cheating the Model)
**Dropped:** `neo`, `class`
* **Why:** `neo` (Near-Earth Object) and `class` are highly correlated with the `pha` target. In a real-world scenario, we wouldn't have these labels beforehand. Keeping them artificially inflates model accuracy.

### 3. Redundant / Duplicate Formats
**Dropped:** `epoch`, `epoch_cal`, `tp_cal`
* **Why:** These are just different datetime formats of features we already have numerically (`epoch_mjd`, `tp`). Neural networks require a single numeric representation.

### 4. Extreme Missingness (>85% Nulls)
**Dropped:** `albedo`, `diameter`, `diameter_sigma`
* **Why:** These columns are missing in over 85% of the dataset.
  1. Imputing 85% of data introduces massive synthetic noise.
  2. `diameter` is scientifically derived using `H` (which we already have and is 100% complete). We lose very little predictive power by dropping it.

---

## Summary of Final Model Inputs
The final preprocessed dataset is optimized for deep learning:
1. **Target:** `pha`
2. **Features used:** All `sigma_*` columns, `H`, and all core orbital parameters (`a, e, q, i, moid`, etc.)
3. **Imputation & Scaling:** Remaining missing values are imputed, and all numerical features are Standard Scaled to ensure stable Neural Network gradient descent.

In [7]:
missing_percent = df.isna().mean() * 100
print(missing_percent.sort_values(ascending=False))

prefix            99.998122
albedo            85.905100
diameter_sigma    85.803068
diameter          85.789714
sigma_per          2.078821
sigma_ad           2.078821
sigma_om           2.078404
sigma_ma           2.078404
sigma_q            2.078404
sigma_a            2.078404
sigma_w            2.078404
sigma_i            2.078404
sigma_tp           2.078404
sigma_e            2.078404
sigma_n            2.078404
moid               2.078300
pha                2.078300
H                  0.653400
moid_ld            0.013250
neo                0.000417
ad                 0.000417
per                0.000417
rms                0.000209
per_y              0.000104
ma                 0.000104
id                 0.000000
full_name          0.000000
pdes               0.000000
orbit_id           0.000000
spkid              0.000000
tp_cal             0.000000
om                 0.000000
w                  0.000000
tp                 0.000000
n                  0.000000
i                  0

In [8]:
drop_cols = [
    'id', 'spkid', 'full_name', 'pdes', 'prefix',
    'orbit_id','equinox',
    'epoch', 'epoch_cal',
    'tp_cal', 'neo', 'class',
    'albedo','diameter','diameter_sigma'
]

df = df.drop(columns=drop_cols)

# Data Type Correction: Converting 'Object' Columns to 'Float64'

When checking the dataset using `df.info()`, we discovered that **20 critically important numerical columns** (such as `H`, `moid`, `per`, and all the `sigma_*` uncertainty columns) were stored as `dtype=object` (text/strings) instead of `float64` (decimals).

**Why did this happen?**
In large datasets (like this NASA asteroid dataset with ~1 million rows), if even **one single row** contains a hidden space (e.g., `" 1683.14"`), a dash (`"-"`), or a missing value represented by a symbol (`"?"`), Pandas panics and casts the entire column as text (`object`).

If we pass `object` data into a `StandardScaler` or a Neural Network, it will immediately crash because it cannot perform mathematics on text.

---
To fix this, we forced Pandas to convert these columns back into numbers using the `pd.to_numeric()` function.

Crucially, we used the `errors='coerce'` argument.
* **What it does:** It takes any valid number (even if it's trapped in a string) and turns it into a clean `float64`.
* **Handling garbage data:** If it encounters a completely invalid character (like a letter or symbol), it turns it into a `NaN` (Null) value. This is perfect, because we can safely handle `NaN` values later using median imputation.

In [9]:
cols_to_fix = [
    'H', 'ma', 'ad', 'per', 'per_y', 'moid', 'moid_ld', 'rms',
    'sigma_e', 'sigma_a', 'sigma_q', 'sigma_i', 'sigma_om',
    'sigma_w', 'sigma_ma', 'sigma_ad', 'sigma_n', 'sigma_tp', 'sigma_per'
]

In [10]:
for col in cols_to_fix:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 958524 entries, 0 to 958523
Data columns (total 29 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   pha        938603 non-null  object 
 1   H          952261 non-null  float64
 2   epoch_mjd  958524 non-null  int64  
 3   e          958524 non-null  float64
 4   a          958524 non-null  float64
 5   q          958524 non-null  float64
 6   i          958524 non-null  float64
 7   om         958524 non-null  float64
 8   w          958524 non-null  float64
 9   ma         958523 non-null  float64
 10  ad         958520 non-null  float64
 11  n          958524 non-null  float64
 12  tp         958524 non-null  float64
 13  per        958520 non-null  float64
 14  per_y      958523 non-null  float64
 15  moid       938603 non-null  float64
 16  moid_ld    958397 non-null  float64
 17  sigma_e    938602 non-null  float64
 18  sigma_a    938602 non-null  float64
 19  sigma_q    938602 non-n

In [11]:
df.drop_duplicates(inplace=True)

In [12]:
x=df.drop('pha',axis=1)
y=df['pha']

In [13]:
print(y.unique())

['N' 'Y' nan]


In [14]:
mask = y.notna()

x = x[mask]
y = y[mask]

y = y.map({
    'N': 0,
    'Y': 1
})

In [15]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.30,random_state=13)

In [16]:
x_train=x_train.fillna(x_train.median())
x_test=x_test.fillna(x_test.median())

In [17]:
print(x_train.isnull().sum().sum(),",", x_test.isnull().sum().sum())

0 , 0


In [18]:
from sklearn.preprocessing import StandardScaler

st=StandardScaler().set_output(transform='pandas')

train_transformed=st.fit_transform(x_train)
test_transformed=st.transform(x_test)

In [19]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

In [20]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [21]:
SEED = 42

torch.cuda.manual_seed(SEED)

In [22]:
class CustomAsteroid(Dataset):

  def __init__(self,features,labels):
    self.features=torch.tensor(features.values,dtype=torch.float32)
    self.labels=torch.tensor(labels.values,dtype=torch.float32)

  def __len__(self):
    return(len(self.labels))

  def __getitem__(self,idx):
    return(self.features[idx],self.labels[idx])

In [23]:
train_dataset=CustomAsteroid(train_transformed,y_train)
test_dataset=CustomAsteroid(test_transformed,y_test)

In [24]:
from torch.utils.data import WeightedRandomSampler
import numpy as np

class_counts = np.bincount(
    y_train.astype(int)
)

print(class_counts)

weights = 1. / class_counts

print(weights)

sample_weights = weights[
    y_train.astype(int)
]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=512,
    sampler=sampler
)

test_dataloader=DataLoader(test_dataset,batch_size=512)

[655566   1456]
[1.52539943e-06 6.86813187e-04]


In [25]:
feature_count=len(x.columns)
print(feature_count)

28


In [26]:
class NeuralNetwork(nn.Module):

  def __init__(self,feature_count):

    super().__init__()

    self.network=nn.Sequential(
            nn.Linear(feature_count, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )


  def forward(self,features):
    self.features=features

    return(self.network(features))

In [27]:
import torch.nn.functional as F

class FocalLoss(nn.Module):

    def __init__(self, alpha=1, gamma=2):

        super().__init__()

        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):

        bce = F.binary_cross_entropy_with_logits(
            inputs,
            targets,
            reduction='none'
        )

        pt = torch.exp(-bce)

        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce

        return focal_loss.mean()

In [28]:
model=NeuralNetwork(feature_count=feature_count).to(device)

criterion = FocalLoss(alpha=3, gamma=2)

optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

In [29]:
epochs = 20

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for features, labels in train_dataloader:

        features=features.to(device)
        labels=labels.to(device)

        labels = labels.unsqueeze(1)

        outputs = model(features)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")

Epoch 1 Loss: 28.5738
Epoch 2 Loss: 20.1178
Epoch 3 Loss: 15.6527
Epoch 4 Loss: 10.7366
Epoch 5 Loss: 10.3553
Epoch 6 Loss: 9.0257
Epoch 7 Loss: 8.7572
Epoch 8 Loss: 8.1563
Epoch 9 Loss: 7.0210
Epoch 10 Loss: 6.4383
Epoch 11 Loss: 6.2319
Epoch 12 Loss: 7.2663
Epoch 13 Loss: 7.3893
Epoch 14 Loss: 6.5631
Epoch 15 Loss: 6.1294
Epoch 16 Loss: 5.7030
Epoch 17 Loss: 5.3733
Epoch 18 Loss: 5.5177
Epoch 19 Loss: 5.1242
Epoch 20 Loss: 4.7647


In [30]:
from sklearn.metrics import (classification_report,confusion_matrix, roc_auc_score)

model.eval()

all_preds = []
all_probs = []
all_labels = []

with torch.no_grad():

    for features, labels in test_dataloader:

        features = features.to(device)
        labels = labels.to(device)

        outputs = model(features)

        probs = torch.sigmoid(outputs)

        preds = (probs>0.8).float()

        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(all_labels, all_preds))

print(confusion_matrix(all_labels, all_preds))

print("ROC AUC:", roc_auc_score(all_labels, all_probs))

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00    280971
         1.0       0.74      0.99      0.84       610

    accuracy                           1.00    281581
   macro avg       0.87      0.99      0.92    281581
weighted avg       1.00      1.00      1.00    281581

[[280754    217]
 [     6    604]]
ROC AUC: 0.9999318405825792
